<figure>
<center>
<img src='https://www.economicas.uba.ar/wp-content/uploads/2020/08/cropped-logo_FCE.png' />
</figure>

#<center>NumPy</center>

# Álgebra Lineal con NumPy

NumPy implementa operaciones de álgebra lineal de forma eficiente.
En lugar de loops, las operaciones matriciales trabajan sobre arrays completos.

Temas:
1. Vectores y producto punto
2. Multiplicación matriz × vector
3. Multiplicación matriz × matriz
4. Transpuesta, inversa y determinante
5. Sistema de ecuaciones lineales

In [1]:
import numpy as np

---
## 1. Vectores y producto punto

Un vector es un array 1D. El producto punto entre dos vectores
multiplica elemento a elemento y suma los resultados:

$$\mathbf{a} \cdot \mathbf{b} = \sum_{i} a_i \, b_i$$

In [ ]:
a = np.array([1, 2, 3])
b = np.array([4, 5, 6])

# Tres formas equivalentes
print( np.dot(a, b) )   # función
print( a @ b )          # operador (más legible)
print( sum(a * b) )     # manual — equivalente pero más lento

32
32
32


El producto punto mide cuánto "apuntan en la misma dirección" dos vectores:
- Resultado alto → vectores similares
- Resultado 0 → vectores perpendiculares (ortogonales)

In [3]:
# Ejemplo: similitud entre perfiles de compra
v_1 = np.array([5, 0, 3, 1])   
v_2 = np.array([4, 0, 2, 1])   
v_3 = np.array([0, 5, 0, 4])   

print("Similitud 1 vs 2:", np.dot(v_1, v_2))
print("Similitud 1 vs 3:", np.dot(v_1, v_3))

Similitud 1 vs 2: 27
Similitud 1 vs 3: 4


---
## 2. Multiplicación matriz × vector

Una matriz transforma un vector: cada fila de la matriz
se multiplica (producto punto) con el vector.

$$\mathbf{y} = A\mathbf{x} \quad \Rightarrow \quad y_i = \sum_j A_{ij}\, x_j$$

In [ ]:
# Ventas por producto en cada sucursal (3 sucursales, 4 productos)
ventas = np.array([
    [10, 5, 8, 3],   
    [ 7, 9, 2, 6],   
    [ 4, 3, 7, 8],   
])

# Precio de cada producto
precios = np.array([100, 250, 80, 150])

# Total facturado por sucursal = ventas @ precios
totales = ventas @ precios

print("Total Norte: $", totales[0])
print("Total Sur:   $", totales[1])
print("Total Centro:$", totales[2])

---
## 3. Multiplicación matriz × matriz

`A @ B` es multiplicación matricial: cada elemento (i,j) del resultado
es el producto punto entre la fila i de A y la columna j de B.

No confundir con `A * B` que multiplica **elemento a elemento**.

In [ ]:
A = np.array([[1, 2],
              [3, 4]])

B = np.array([[5, 6],
              [7, 8]])

print("A @ B (matricial):")
print(A @ B)

print("\nA * B (elemento a elemento):")
print(A * B)

**Condición de forma**: para que `A @ B` sea válido,
el número de columnas de A debe coincidir con el número de filas de B.

$$A_{(m \times n)} \; @ \; B_{(n \times p)} \; \rightarrow \; \text{resultado}_{(m \times p)}$$

In [ ]:
# Aplicación: componer dos transformaciones
# En lugar de aplicarlas de a una, las combinamos en una sola matriz

T1 = np.array([[2, 0], [0, 2]])    # escalar × 2
T2 = np.array([[0, -1], [1, 0]])   # rotar 90°

T_compuesta = T2 @ T1   

v = np.array([1, 0])
print("Transformación compuesta:  ", T_compuesta @ v)

Transformación compuesta:   [0 2]
Paso a paso (mismo resultado): [0 2]


---
## 4. Transpuesta, inversa y determinante

Operaciones fundamentales sobre matrices cuadradas.

In [5]:
A = np.array([[1, 2, 3],
              [4, 5, 6]])

# Transpuesta: filas ↔ columnas
print("A shape:", A.shape)
print("A.T shape:", A.T.shape)
print(A.T)

A shape: (2, 3)
A.T shape: (3, 2)
[[1 4]
 [2 5]
 [3 6]]


In [7]:
# Inversa y determinante — solo para matrices cuadradas
M = np.array([[2, 1],
              [5, 3]])

print("Determinante:", np.linalg.det(M))

print("\nInversa:")
print(np.linalg.inv(M))

# Verificación: M @ M⁻¹ debe dar la identidad
print("\nM @ M⁻¹ (debe ser identidad):")
print(np.round(M @ np.linalg.inv(M)).astype(int))

Determinante: 1.0000000000000009

Inversa:
[[ 3. -1.]
 [-5.  2.]]

M @ M⁻¹ (debe ser identidad):
[[1 0]
 [0 1]]


Si el determinante es **0**, la matriz es **singular** y no tiene inversa.
Esto ocurre cuando las filas (o columnas) son linealmente dependientes —
una fila es combinación lineal de las otras.

In [8]:
# Matriz singular — segunda fila = 2 × primera fila
S = np.array([[1, 2],
              [2, 4]])

print("Determinante:", np.linalg.det(S))  # ≈ 0

try:
    np.linalg.inv(S)
except np.linalg.LinAlgError as e:
    print("Error:", e)

Determinante: 0.0
Error: Singular matrix


---
## 5. Sistema de ecuaciones lineales

Un sistema $A\mathbf{x} = \mathbf{b}$ puede resolverse con `np.linalg.solve`
cuando A es cuadrada e inversible.
Es equivalente a $\mathbf{x} = A^{-1}\mathbf{b}$ pero numéricamente más estable.

**Problema**: una fábrica produce 3 productos (A, B, C).
Cada uno consume horas de máquina, horas de mano de obra y kg de material.
Conocemos el consumo total de cada recurso. ¿Cuántas unidades de cada producto se fabricaron?

| Recurso     | Prod A | Prod B | Prod C | Total |
|-------------|--------|--------|--------|-------|
| Máquina     |   2    |   1    |   3    |  420  |
| Mano de obra|   4    |   3    |   2    |  680  |
| Material    |   1    |   2    |   1    |  290  |

El sistema queda:

$$2x_A + x_B + 3x_C = 420$$
$$4x_A + 3x_B + 2x_C = 680$$
$$x_A + 2x_B + x_C = 290$$

In [9]:
# Coeficientes: cada fila es una ecuación, cada columna un producto
A = np.array([
    [2, 1, 3],   # horas máquina por unidad
    [4, 3, 2],   # horas mano de obra por unidad
    [1, 2, 1],   # kg material por unidad
])

# Recursos totales consumidos
b = np.array([420, 680, 290])

# Resolver el sistema
x = np.linalg.solve(A, b)

print(f"Producto A: {x[0]:.0f} unidades")
print(f"Producto B: {x[1]:.0f} unidades")
print(f"Producto C: {x[2]:.0f} unidades")

Producto A: 86 unidades
Producto B: 73 unidades
Producto C: 58 unidades


In [10]:
# Verificación: A @ x debe reproducir b
print("Verificación (A @ x):", np.round(A @ x).astype(int))
print("Vector b original:   ", b)

Verificación (A @ x): [420 680 290]
Vector b original:    [420 680 290]
